In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

In [3]:
INPUT_FILE = "/Volumes/ORICO/Dissertation/Dissertation_data.xlsx"
SHEET_NAME = "Dissertation_data"
OUTPUT_DIR = "/Volumes/ORICO/Dissertation/output"

PRIMARY_INDUSTRY_SOURCE = "naics"   # "naics" or "sic"
WINSORIZE = True
WINSOR_LOWER = 0.01
WINSOR_UPPER = 0.99

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_FILE)
print("Output dir:", OUTPUT_DIR)

Input: /Volumes/ORICO/Dissertation/Dissertation_data.xlsx
Output dir: /Volumes/ORICO/Dissertation/output


In [4]:
def first_n_digits(value, n=3):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if "." in s:
        s = s.split(".")[0]
    digits = "".join(ch for ch in s if ch.isdigit())
    if len(digits) < n:
        return np.nan
    return digits[:n]

def safe_divide(num, den):
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")
    return np.where((den == 0) | pd.isna(den), np.nan, num / den)

def winsorize_series(s, lower=0.01, upper=0.99):
    s = pd.to_numeric(s, errors="coerce")
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lower=lo, upper=hi)

def add_lag(df, group_col, var_list, lag=1):
    for v in var_list:
        df[f"{v}_L{lag}"] = df.groupby(group_col)[v].shift(lag)
    return df

def add_log1p_if_nonnegative(df, var_list):
    for v in var_list:
        s = pd.to_numeric(df[v], errors="coerce")
        if (s.dropna() >= 0).all():
            df[f"ln_{v}"] = np.log1p(s)
        else:
            df[f"ln_{v}"] = np.nan
    return df

def add_asinh(df, var_list):
    for v in var_list:
        s = pd.to_numeric(df[v], errors="coerce")
        df[f"asinh_{v}"] = np.arcsinh(s)
    return df

In [5]:
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
df.columns = [c.strip().lower() for c in df.columns]

print("Raw shape:", df.shape)
print(df.columns.tolist())
df.head()

Raw shape: (1797, 980)
['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'gvkey', 'datadate', 'conm', 'tic', 'cusip', 'cik', 'exchg', 'fyr', 'fic', 'add1', 'add2', 'add3', 'add4', 'addzip', 'busdesc', 'city', 'conml', 'county', 'dldte', 'dlrsn', 'ein', 'fax', 'fyrc', 'ggroup', 'gind', 'gsector', 'gsubind', 'idbflag', 'incorp', 'ipodate', 'loc', 'naics', 'phone', 'prican', 'prirow', 'priusa', 'sic', 'spcindcd', 'spcseccd', 'spcsrc', 'state', 'stko', 'weburl', 'acctchg', 'acctstd', 'acqmeth', 'adrr', 'ajex', 'ajp', 'apdedate', 'bspr', 'compst', 'curncd', 'currtr', 'curuscn', 'fdate', 'final', 'fyear', 'ismod', 'ltcm', 'ogm', 'pdate', 'pddur', 'scf', 'src', 'stalt', 'udpl', 'upd', 'acco', 'acdo', 'aco', 'acodo', 'acominc', 'acox', 'acoxar', 'act', 'aedi', 'aldo', 'ao', 'aocidergl', 'aociother', 'aocipen', 'aocisecgl', 'aodo', 'aox', 'ap', 'apb', 'apc', 'apofs', 'arb', 'arc', 'artfs', 'at', 'bast', 'bkvlps', 'ca', 'caps', 'cb', 'ceq', 'ceql', 'ceqt', 'ch', 'che', 'chs', 'cld2', 'cld3', 'c

,costat,curcd,datafmt,indfmt,consol,gvkey,datadate,conm,tic,cusip,...,dvpsx_f,mkvalt,naicsh,prcc_c,prcc_f,prch_c,prch_f,prcl_c,prcl_f,sich
0,I,USD,STD,INDL,C,2163,2010-03-31,BENIHANA INC,BNHN,082047101,...,0.00,102.0678,722110.0,3.79,6.50,8.38,8.38,1.56,2.43,5812.0
1,I,USD,STD,INDL,C,2163,2011-03-31,BENIHANA INC,BNHN,082047101,...,0.00,135.5790,722110.0,8.18,8.45,8.51,9.50,3.67,4.82,5812.0
2,I,USD,STD,INDL,C,2163,2012-03-31,BENIHANA INC,BNHN,082047101,...,0.08,233.8560,722110.0,10.23,13.05,10.81,13.75,6.69,6.69,5812.0
3,I,USD,STD,INDL,C,2791,2010-12-31,CARROLS CORP,TAST1,145744009,...,0.00,NaN,722211.0,NaN,NaN,NaN,NaN,NaN,NaN,5812.0
4,A,USD,STD,INDL,C,3007,2010-06-30,BRINKER INTL INC,EAT,109641100,...,0.47,1468.7311,722110.0,20.88,14.46,22.56,21.12,13.96,12.39,5812.0


In [6]:
df["datadate"] = pd.to_datetime(df["datadate"], errors="coerce")

if "fyear" in df.columns:
    df["year"] = pd.to_numeric(df["fyear"], errors="coerce")
else:
    df["year"] = df["datadate"].dt.year

df["gvkey"] = df["gvkey"].astype(str).str.strip()

df = df.loc[df["gvkey"].notna() & df["year"].notna()].copy()
df = df.sort_values(["gvkey", "year", "datadate"])
df = df.drop_duplicates(subset=["gvkey", "year"], keep="last").copy()

df["naics3"] = df["naics"].apply(lambda x: first_n_digits(x, 3))
df["sic3"] = df["sic"].apply(lambda x: first_n_digits(x, 3))

if PRIMARY_INDUSTRY_SOURCE == "naics":
    df["industry3"] = df["naics3"].fillna(df["sic3"])
else:
    df["industry3"] = df["sic3"].fillna(df["naics3"])

df = df[df["industry3"].isin(["721", "722"])].copy()
df = df.sort_values(["gvkey", "year"]).reset_index(drop=True)

print("Filtered shape (721/722 only):", df.shape)
print(df["industry3"].value_counts(dropna=False).sort_index())
df[["gvkey", "year", "industry3"]].head()

Filtered shape (721/722 only): (1769, 984)
industry3
721     580
722    1189
Name: count, dtype: int64


,gvkey,year,industry3
0,102089,2010,722
1,102089,2011,722
2,102089,2012,722
3,102089,2013,722
4,102089,2014,722


In [7]:
numeric_cols = [
    "sale", "ni", "oibdp", "capx", "ppent", "emp", "xsga", "intan",
    "at", "ceq", "prcc_f", "csho", "mkvalt", "dltt", "dlc", "act",
    "lct", "re", "ebit", "che", "lt", "seq", "ib", "txditc"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols].head()

,sale,ni,oibdp,capx,ppent,emp,xsga,intan,at,ceq,...,dlc,act,lct,re,ebit,che,lt,seq,ib,txditc
0,19396.467,519.594,1267.859,299.815,674.582,379.137,2091.079,6556.535,14958.960,3438.973,...,265.514,6944.006,7273.040,1523.209,976.937,2681.815,11479.334,3438.973,519.594,154.988
1,23117.308,649.711,1568.813,348.625,739.028,391.148,2407.243,6878.865,16435.805,3651.921,...,252.105,7991.008,8496.659,1712.874,1220.188,2995.008,12740.666,3651.921,649.711,216.090
2,22937.241,660.345,1612.501,387.402,721.977,421.391,2349.570,7036.133,16087.262,3816.165,...,189.928,7423.536,7924.140,2154.612,1189.880,2591.069,12227.074,3816.165,660.345,202.506
3,24276.681,579.304,1576.922,318.024,712.584,427.921,2458.415,7034.787,16637.517,3896.779,...,992.339,7895.167,9084.126,2121.917,1227.228,2746.087,12691.913,3896.779,579.304,201.899
4,23691.040,644.350,1526.715,322.175,729.825,419.000,2438.010,7225.925,19038.570,4193.535,...,1338.670,10028.190,9766.505,2447.215,1195.335,4610.390,14802.955,4193.535,644.350,194.620


In [8]:
# Main DVs
df["gop"] = df["oibdp"]
df["ni_dv"] = df["ni"]

# Market value
df["mve_calc"] = df["prcc_f"] * df["csho"]
df["mve"] = df["mkvalt"]
df["mve"] = np.where(pd.isna(df["mve"]), df["mve_calc"], df["mve"])

# Tobin's Q
df["tobinq"] = safe_divide(df["mve"] + df["at"] - df["ceq"], df["at"])

# Backup DVs
df["roa"] = safe_divide(df["ni"], df["at"])
df["ros"] = safe_divide(df["ni"], df["sale"])
df["operating_margin"] = safe_divide(df["oibdp"], df["sale"])
df["sales_growth"] = df.groupby("gvkey")["sale"].pct_change()

df[["gop", "ni_dv", "tobinq", "roa", "ros", "operating_margin", "sales_growth"]].describe()

,gop,ni_dv,tobinq,roa,ros,operating_margin,sales_growth
count,1709.000000,1713.000000,1520.000000,1710.000000,1690.000000,1686.000000,1511.000000
mean,440.393930,182.406511,9.164365,-0.601107,1.191662,-0.075996,inf
std,1205.087517,772.928604,179.177541,15.956333,49.519716,7.312829,NaN
min,-3172.290000,-4584.716000,0.279389,-608.000000,-45.000000,-296.000000,-1.000000
25%,9.470000,-2.525000,1.211718,-0.023878,-0.028513,0.070354,-0.006415
50%,58.238000,10.025000,1.607638,0.029305,0.033314,0.126181,0.059112
75%,341.117000,100.072000,2.515116,0.072436,0.087047,0.224722,0.158897
max,13350.000000,8468.800000,6638.800000,7.842167,2033.000000,0.986257,inf


In [9]:
# Raw versions
df["capx_raw"] = df["capx"]
df["ppent_raw"] = df["ppent"]
df["emp_raw"] = df["emp"]
df["xsga_raw"] = df["xsga"]
df["intan_raw"] = df["intan"]

# Scaled by total assets
df["capx_at"] = safe_divide(df["capx"], df["at"])
df["ppent_at"] = safe_divide(df["ppent"], df["at"])
df["xsga_at"] = safe_divide(df["xsga"], df["at"])
df["intan_at"] = safe_divide(df["intan"], df["at"])

# Employee scaling alternatives
df["emp_at"] = safe_divide(df["emp"], df["at"])
df["emp_sale"] = safe_divide(df["emp"], df["sale"])

df[[
    "capx_raw", "ppent_raw", "emp_raw", "xsga_raw", "intan_raw",
    "capx_at", "ppent_at", "emp_at", "emp_sale", "xsga_at", "intan_at"
]].describe()

,capx_raw,ppent_raw,emp_raw,xsga_raw,intan_raw,capx_at,ppent_at,emp_at,emp_sale,xsga_at,intan_at
count,1711.000000,1715.000000,1558.000000,1598.000000,1674.000000,1708.000000,1712.000000,1555.000000,1541.000000,1595.000000,1671.000000
mean,138.499987,1556.620508,34.937060,279.644509,947.920251,0.085635,0.481696,0.018360,0.038564,0.688691,0.196469
std,331.247437,4084.377498,79.973706,584.422076,2741.553924,0.865954,0.276828,0.027376,0.803317,14.946566,0.242393
min,-0.145000,0.000000,0.000000,0.000000,0.000000,-0.005658,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.717000,49.528500,2.000000,15.872000,4.363500,0.018285,0.255110,0.003436,0.007245,0.068167,0.011795
50%,31.405000,250.000000,7.081000,72.209000,67.128500,0.045186,0.533553,0.010131,0.013903,0.113845,0.091443
75%,112.090500,1084.498000,23.494250,247.137750,329.689250,0.088080,0.708410,0.027588,0.020298,0.197435,0.296102
max,3049.200000,38785.900000,539.000000,5513.000000,18219.000000,35.714286,1.000000,0.500000,31.500000,581.500000,1.000000


In [10]:
# Size
df["size_ln_at"] = np.where(df["at"] > 0, np.log(df["at"]), np.nan)

# Debt and leverage
df["total_debt"] = df[["dltt", "dlc"]].sum(axis=1, min_count=1)
df["lev"] = safe_divide(df["total_debt"], df["at"])

# Liquidity
df["liquidity"] = safe_divide(df["act"], df["lct"])

# Asset turnover
df["asset_turnover"] = safe_divide(df["sale"], df["at"])

# Working capital
df["wc"] = df["act"] - df["lct"]

# Altman Z-score
total_liabilities_for_z = df["lt"] if "lt" in df.columns else df["total_debt"]

df["z_score"] = (
    1.2 * safe_divide(df["wc"], df["at"])
    + 1.4 * safe_divide(df["re"], df["at"])
    + 3.3 * safe_divide(df["ebit"], df["at"])
    + 0.6 * safe_divide(df["mve"], total_liabilities_for_z)
    + 1.0 * safe_divide(df["sale"], df["at"])
)

df["distress_dummy"] = np.where(
    df["z_score"] < 1.81, 1,
    np.where(df["z_score"].notna(), 0, np.nan)
)

df[[
    "size_ln_at", "lev", "liquidity", "asset_turnover",
    "z_score", "distress_dummy"
]].describe()

,size_ln_at,lev,liquidity,asset_turnover,z_score,distress_dummy
count,1713.000000,1713.000000,1602.000000,1710.000000,1401.000000,1401.000000
mean,6.321491,0.544823,1.239797,1.003779,-7.697118,0.446110
std,2.355462,1.125189,1.563547,0.953335,208.176321,0.497265
min,-6.214608,0.000000,0.000000,0.000000,-5821.590304,0.000000
25%,5.016670,0.204895,0.551785,0.436331,1.009987,0.000000
50%,6.496856,0.442318,0.946561,0.758211,2.080678,0.000000
75%,7.918284,0.659729,1.443751,1.431969,3.641847,1.000000
max,10.935725,36.500000,27.788059,24.666667,325.818120,1.000000


In [11]:
industry_year_sales = df.groupby(["industry3", "year"])["sale"].transform("sum")
df["market_share"] = safe_divide(df["sale"], industry_year_sales)
df["market_share_sq"] = df["market_share"] ** 2
df["firm_hhi"] = df.groupby(["industry3", "year"])["market_share_sq"].transform("sum")

df[["gvkey", "year", "industry3", "sale", "market_share", "firm_hhi"]].head(20)

,gvkey,year,industry3,sale,market_share,firm_hhi
0,102089,2010,722,19396.467,0.138556,0.077362
1,102089,2011,722,23117.308,0.151490,0.081876
2,102089,2012,722,22937.241,0.136361,0.077811
3,102089,2013,722,24276.681,0.153849,0.085320
4,102089,2014,722,23691.040,0.142953,0.079843
5,102089,2015,722,22180.911,0.127440,0.070435
6,102089,2016,722,22565.077,0.134103,0.073006
7,102089,2017,722,24618.201,0.148644,0.078267
8,102089,2018,722,23688.619,0.139178,0.076524
9,102089,2019,722,24147.814,0.137376,0.076353


In [12]:
lag_vars = [
    "capx_raw", "ppent_raw", "emp_raw", "xsga_raw", "intan_raw",
    "capx_at", "ppent_at", "emp_at", "emp_sale", "xsga_at", "intan_at",
    "size_ln_at", "lev", "liquidity", "asset_turnover", "z_score",
    "distress_dummy", "firm_hhi",
    "gop", "ni_dv", "tobinq", "roa", "ros", "operating_margin", "sales_growth"
]

df = add_lag(df, "gvkey", lag_vars, lag=1)

lag_cols_preview = [c for c in df.columns if c.endswith("_L1")]
print("Number of lagged columns:", len(lag_cols_preview))
df[lag_cols_preview[:15]].head()

Number of lagged columns: 25


,capx_raw_L1,ppent_raw_L1,emp_raw_L1,xsga_raw_L1,intan_raw_L1,capx_at_L1,ppent_at_L1,emp_at_L1,emp_sale_L1,xsga_at_L1,intan_at_L1,size_ln_at_L1,lev_L1,liquidity_L1,asset_turnover_L1
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,299.815,674.582,379.137,2091.079,6556.535,0.020043,0.045096,0.025345,0.019547,0.139788,0.438302,9.613066,0.232951,0.954760,1.296645
2,348.625,739.028,391.148,2407.243,6878.865,0.021211,0.044965,0.023799,0.016920,0.146463,0.418529,9.707217,0.213603,0.940488,1.406521
3,387.402,721.977,421.391,2349.570,7036.133,0.024081,0.044879,0.026194,0.018371,0.146052,0.437373,9.685783,0.211181,0.936825,1.425801
4,318.024,712.584,427.921,2458.415,7034.787,0.019115,0.042830,0.025720,0.017627,0.147763,0.422827,9.719415,0.209946,0.869117,1.459153


In [13]:
log_candidates = [
    "gop", "capx_raw", "ppent_raw", "emp_raw", "xsga_raw", "intan_raw",
    "capx_at", "ppent_at", "emp_at", "emp_sale", "xsga_at", "intan_at"
]
df = add_log1p_if_nonnegative(df, log_candidates)

asinh_candidates = [
    "gop", "ni_dv", "tobinq", "roa", "ros", "operating_margin",
    "capx_raw", "ppent_raw", "emp_raw", "xsga_raw", "intan_raw",
    "capx_at", "ppent_at", "emp_at", "emp_sale", "xsga_at", "intan_at",
    "sales_growth"
]
df = add_asinh(df, asinh_candidates)

transform_cols = [c for c in df.columns if c.startswith("ln_") or c.startswith("asinh_")]
print("Number of transformed columns:", len(transform_cols))
df[transform_cols[:15]].head()

Number of transformed columns: 30


,ln_gop,ln_capx_raw,ln_ppent_raw,ln_emp_raw,ln_xsga_raw,ln_intan_raw,ln_capx_at,ln_ppent_at,ln_emp_at,ln_emp_sale,ln_xsga_at,ln_intan_at,asinh_gop,asinh_ni_dv,asinh_tobinq
0,NaN,NaN,6.515575,5.940532,7.645914,8.788370,NaN,0.044108,0.025029,0.019358,0.130842,0.363463,7.838232,6.946196,1.123643
1,NaN,NaN,6.606688,5.971639,7.786653,8.836354,NaN,0.043983,0.023520,0.016779,0.136682,0.349621,8.051222,7.169675,1.173535
2,NaN,NaN,6.583377,6.045931,7.762413,8.858956,NaN,0.043901,0.025857,0.018205,0.136323,0.362817,8.078689,7.185910,1.197430
3,NaN,NaN,6.570300,6.061273,7.807679,8.858765,NaN,0.041938,0.025395,0.017473,0.137815,0.352646,8.056377,7.054975,1.231664
4,NaN,NaN,6.594174,6.040255,7.799347,8.885569,NaN,0.037618,0.021769,0.017531,0.120496,0.321751,8.024021,7.161390,1.231507


In [14]:
if WINSORIZE:
    winsor_vars = [
        "gop", "ni_dv", "tobinq", "roa", "ros", "operating_margin", "sales_growth",
        "capx_at", "ppent_at", "emp_at", "emp_sale", "xsga_at", "intan_at",
        "lev", "liquidity", "asset_turnover", "z_score", "firm_hhi"
    ]

    for v in winsor_vars:
        if v in df.columns:
            df[f"{v}_w"] = winsorize_series(df[v], WINSOR_LOWER, WINSOR_UPPER)

winsor_cols = [c for c in df.columns if c.endswith("_w")]
print("Number of winsorized columns:", len(winsor_cols))
df[winsor_cols[:15]].head()

Number of winsorized columns: 18


,gop_w,ni_dv_w,tobinq_w,roa_w,ros_w,operating_margin_w,sales_growth_w,capx_at_w,ppent_at_w,emp_at_w,emp_sale_w,xsga_at_w,intan_at_w,lev_w,liquidity_w
0,1267.859,519.594,1.375474,0.034735,0.026788,0.065365,NaN,0.020043,0.045096,0.025345,0.019547,0.139788,0.438302,0.232951,0.954760
1,1568.813,649.711,1.462066,0.039530,0.028105,0.067863,0.191831,0.021211,0.044965,0.023799,0.016920,0.146463,0.418529,0.213603,0.940488
2,1612.501,660.345,1.504812,0.041048,0.028789,0.070301,-0.007789,0.024081,0.044879,0.026194,0.018371,0.146052,0.437373,0.211181,0.936825
3,1576.922,579.304,1.567560,0.034819,0.023863,0.064956,0.058396,0.019115,0.042830,0.025720,0.017627,0.147763,0.422827,0.209946,0.869117
4,1526.715,644.350,1.567268,0.033844,0.027198,0.064443,-0.024124,0.016922,0.038334,0.022008,0.017686,0.128056,0.379541,0.270272,1.026794


In [15]:
missingness = pd.DataFrame({
    "variable": df.columns,
    "n_missing": df.isna().sum().values,
    "pct_missing": (df.isna().mean() * 100).round(2).values
}).sort_values(["pct_missing", "variable"], ascending=[False, True])

missingness.head(30)

,variable,n_missing,pct_missing
73,acco,1769,100.0
79,acoxar,1769,100.0
856,acqlntal,1769,100.0
857,acqniintc,1769,100.0
16,add3,1769,100.0
17,add4,1769,100.0
456,adpac,1769,100.0
81,aedi,1769,100.0
783,afudcc,1769,100.0
784,afudci,1769,100.0


In [16]:
firm_year_counts = (
    df.groupby("gvkey")
      .agg(
          first_year=("year", "min"),
          last_year=("year", "max"),
          n_years=("year", "nunique")
      )
      .reset_index()
      .sort_values(["n_years", "gvkey"], ascending=[False, True])
)

firm_year_counts.head(20)

,gvkey,first_year,last_year,n_years
14,133767,2009,2024,16
132,3007,2010,2025,16
133,31846,2009,2024,16
1,106660,2010,2024,15
7,117036,2010,2024,15
8,11872,2010,2024,15
10,12544,2010,2024,15
12,13092,2010,2024,15
16,13759,2010,2024,15
20,14418,2010,2024,15


In [17]:
id_vars = ["gvkey", "conm", "tic", "year", "datadate", "industry3", "naics", "sic"]

keep_cols = id_vars + [
    # DVs
    "gop", "ni_dv", "tobinq", "roa", "ros", "operating_margin", "sales_growth",
    # IVs raw
    "capx_raw", "ppent_raw", "emp_raw", "xsga_raw", "intan_raw",
    # IVs scaled
    "capx_at", "ppent_at", "emp_at", "emp_sale", "xsga_at", "intan_at",
    # controls
    "size_ln_at", "lev", "liquidity", "asset_turnover", "z_score",
    "distress_dummy", "firm_hhi", "market_share"
]

keep_cols += [f"{v}_L1" for v in lag_vars]
keep_cols += [c for c in df.columns if c.startswith("ln_")]
keep_cols += [c for c in df.columns if c.startswith("asinh_")]
keep_cols += [c for c in df.columns if c.endswith("_w")]

keep_cols = [c for c in keep_cols if c in df.columns]
final_df = df[keep_cols].copy()

print(final_df.shape)
final_df.head()

(1769, 107)


,gvkey,conm,tic,year,datadate,industry3,naics,sic,gop,ni_dv,...,ppent_at_w,emp_at_w,emp_sale_w,xsga_at_w,intan_at_w,lev_w,liquidity_w,asset_turnover_w,z_score_w,firm_hhi_w
0,102089,SODEXO S A,SDXAY,2010,2010-08-31,722,722310,5812,1267.859,519.594,...,0.045096,0.025345,0.019547,0.139788,0.438302,0.232951,0.954760,1.296645,2.101643,0.077362
1,102089,SODEXO S A,SDXAY,2011,2011-08-31,722,722310,5812,1568.813,649.711,...,0.044965,0.023799,0.016920,0.146463,0.418529,0.213603,0.940488,1.406521,2.290123,0.081876
2,102089,SODEXO S A,SDXAY,2012,2012-08-31,722,722310,5812,1612.501,660.345,...,0.044879,0.026194,0.018371,0.146052,0.437373,0.211181,0.936825,1.425801,2.405823,0.077811
3,102089,SODEXO S A,SDXAY,2013,2013-08-31,722,722310,5812,1576.922,579.304,...,0.042830,0.025720,0.017627,0.147763,0.422827,0.209946,0.869117,1.459153,2.425986,0.085320
4,102089,SODEXO S A,SDXAY,2014,2014-08-31,722,722310,5812,1526.715,644.350,...,0.038334,0.022008,0.017686,0.128056,0.379541,0.270272,1.026794,1.244371,2.255734,0.079843


In [18]:
base_required = id_vars + [
    "size_ln_at_L1", "lev_L1", "liquidity_L1", "asset_turnover_L1",
    "z_score_L1", "distress_dummy_L1", "firm_hhi_L1",
    "capx_at_L1", "emp_at_L1", "xsga_at_L1", "intan_at_L1"
]

modeldata_gop = final_df.dropna(
    subset=[c for c in base_required + ["gop"] if c in final_df.columns]
).copy()

modeldata_ni = final_df.dropna(
    subset=[c for c in base_required + ["ni_dv"] if c in final_df.columns]
).copy()

modeldata_tq = final_df.dropna(
    subset=[c for c in base_required + ["tobinq"] if c in final_df.columns]
).copy()

print("modeldata_gop:", modeldata_gop.shape)
print("modeldata_ni :", modeldata_ni.shape)
print("modeldata_tq :", modeldata_tq.shape)

modeldata_gop: (1065, 107)
modeldata_ni : (1065, 107)
modeldata_tq : (1062, 107)


In [19]:
final_csv = f"{OUTPUT_DIR}/dissertation_analysis_ready_full.csv"
final_xlsx = f"{OUTPUT_DIR}/dissertation_analysis_ready_full.xlsx"
gop_csv = f"{OUTPUT_DIR}/modeldata_gop.csv"
ni_csv = f"{OUTPUT_DIR}/modeldata_ni.csv"
tq_csv = f"{OUTPUT_DIR}/modeldata_tobinq.csv"
miss_csv = f"{OUTPUT_DIR}/variable_missingness_summary.csv"
firmyear_csv = f"{OUTPUT_DIR}/firm_year_counts.csv"

final_df.to_csv(final_csv, index=False)
final_df.to_excel(final_xlsx, index=False)

modeldata_gop.to_csv(gop_csv, index=False)
modeldata_ni.to_csv(ni_csv, index=False)
modeldata_tq.to_csv(tq_csv, index=False)

missingness.to_csv(miss_csv, index=False)
firm_year_counts.to_csv(firmyear_csv, index=False)

print("Saved:")
print(final_csv)
print(final_xlsx)
print(gop_csv)
print(ni_csv)
print(tq_csv)
print(miss_csv)
print(firmyear_csv)

Saved:
/Volumes/ORICO/Dissertation/output/dissertation_analysis_ready_full.csv
/Volumes/ORICO/Dissertation/output/dissertation_analysis_ready_full.xlsx
/Volumes/ORICO/Dissertation/output/modeldata_gop.csv
/Volumes/ORICO/Dissertation/output/modeldata_ni.csv
/Volumes/ORICO/Dissertation/output/modeldata_tobinq.csv
/Volumes/ORICO/Dissertation/output/variable_missingness_summary.csv
/Volumes/ORICO/Dissertation/output/firm_year_counts.csv


In [20]:
print("Final columns:", len(final_df.columns))
print("Final rows:", len(final_df))
final_df.sample(min(5, len(final_df)))

Final columns: 107
Final rows: 1769


,gvkey,conm,tic,year,datadate,industry3,naics,sic,gop,ni_dv,...,ppent_at_w,emp_at_w,emp_sale_w,xsga_at_w,intan_at_w,lev_w,liquidity_w,asset_turnover_w,z_score_w,firm_hhi_w
1053,24113,PANERA BREAD CO,PNRA,2013,2013-12-31,722,722513,5812,416.279,196.169,...,0.566882,0.033958,0.016813,0.111045,0.171723,0.005206,0.997992,2.019713,10.228651,0.085320
1417,3708,WENDY'S CO,WEN,2021,2021-12-31,722,722511,5812,486.580,200.392,...,0.384947,0.002842,0.007644,0.156573,0.403041,0.755952,1.388649,0.371859,1.387796,0.076889
499,165914,CHIPOTLE MEXICAN GRILL INC,CMG,2019,2019-12-31,722,722513,5812,723.072,350.158,...,0.776584,0.016260,0.014858,0.082165,0.004298,0.558616,1.608484,1.094379,6.393707,0.076353
1004,22672,SHAKE SHACK INC,SHAK,2018,2018-12-31,722,722513,5812,64.187,15.179,...,0.428895,0.009993,0.013283,0.102271,0.001898,0.034144,1.686779,0.752311,3.479855,0.076524
123,12544,ST JOE CO,JOE,2018,2018-12-31,721,721110,7011,38.402,32.369,...,0.416809,0.000061,0.000481,NaN,0.000000,0.282617,NaN,0.126614,NaN,0.090312


In [21]:
final_df["gvkey"].nunique()

198

In [22]:
print("GOP rows:", modeldata_gop.shape)
print("NI rows:", modeldata_ni.shape)
print("TQ rows:", modeldata_tq.shape)

print("GOP firms:", modeldata_gop["gvkey"].nunique())
print("NI firms:", modeldata_ni["gvkey"].nunique())
print("TQ firms:", modeldata_tq["gvkey"].nunique())

GOP rows: (1065, 107)
NI rows: (1065, 107)
TQ rows: (1062, 107)
GOP firms: 138
NI firms: 138
TQ firms: 136


In [25]:
print(modeldata_gop.columns.tolist())

['gvkey', 'conm', 'tic', 'year', 'datadate', 'industry3', 'naics', 'sic', 'gop', 'ni_dv', 'tobinq', 'roa', 'ros', 'operating_margin', 'sales_growth', 'capx_raw', 'ppent_raw', 'emp_raw', 'xsga_raw', 'intan_raw', 'capx_at', 'ppent_at', 'emp_at', 'emp_sale', 'xsga_at', 'intan_at', 'size_ln_at', 'lev', 'liquidity', 'asset_turnover', 'z_score', 'distress_dummy', 'firm_hhi', 'market_share', 'capx_raw_L1', 'ppent_raw_L1', 'emp_raw_L1', 'xsga_raw_L1', 'intan_raw_L1', 'capx_at_L1', 'ppent_at_L1', 'emp_at_L1', 'emp_sale_L1', 'xsga_at_L1', 'intan_at_L1', 'size_ln_at_L1', 'lev_L1', 'liquidity_L1', 'asset_turnover_L1', 'z_score_L1', 'distress_dummy_L1', 'firm_hhi_L1', 'gop_L1', 'ni_dv_L1', 'tobinq_L1', 'roa_L1', 'ros_L1', 'operating_margin_L1', 'sales_growth_L1', 'ln_gop', 'ln_capx_raw', 'ln_ppent_raw', 'ln_emp_raw', 'ln_xsga_raw', 'ln_intan_raw', 'ln_capx_at', 'ln_ppent_at', 'ln_emp_at', 'ln_emp_sale', 'ln_xsga_at', 'ln_intan_at', 'asinh_gop', 'asinh_ni_dv', 'asinh_tobinq', 'asinh_roa', 'asinh_ros

In [26]:
import statsmodels.formula.api as smf

baseline_gop = smf.ols(
    formula="""
    gop_w ~ 
        capx_at_L1
        + ppent_at_L1
        + emp_at_L1
        + xsga_at_L1
        + intan_at_L1
        + size_ln_at_L1
        + lev_w
        + liquidity_w
        + asset_turnover_w
        + firm_hhi_w
        + C(year)
    """,
    data=modeldata_gop
).fit(cov_type="HC3")

print(baseline_gop.summary())

                            OLS Regression Results                            
Dep. Variable:                  gop_w   R-squared:                       0.442
Model:                            OLS   Adj. R-squared:                  0.429
Method:                 Least Squares   F-statistic:                     17.35
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           2.33e-62
Time:                        20:49:09   Log-Likelihood:                -8610.0
No. Observations:                1064   AIC:                         1.727e+04
Df Residuals:                    1038   BIC:                         1.740e+04
Df Model:                          25                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept         -172.9452    345.523  

In [29]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = modeldata_gop[[
    "capx_at_L1",
    "ppent_at_L1",
    "emp_at_L1",
    "xsga_at_L1",
    "intan_at_L1",
    "size_ln_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w"
]].dropna()

vif = pd.DataFrame()
vif["Variable"] = X.columns
vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print(vif)

           Variable         VIF
0        capx_at_L1  194.307771
1       ppent_at_L1    9.620087
2         emp_at_L1    2.682874
3        xsga_at_L1  197.063622
4       intan_at_L1    3.021444
5     size_ln_at_L1   14.354421
6             lev_w    2.632567
7       liquidity_w    3.597604
8  asset_turnover_w    4.497963
9        firm_hhi_w   29.030431


In [68]:
import statsmodels.formula.api as smf
import pandas as pd

fe_vars = [
    "gop_w", "gop_L1",
    "capx_at_L1", "ppent_at_L1", "emp_at_L1", "xsga_at_L1", "intan_at_L1",
    "lev_w", "size_ln_at_L1", "liquidity_w", "asset_turnover_w", "firm_hhi_w",
    "year", "gvkey"
]

fe_data = modeldata_gop[fe_vars].dropna().copy()

print(fe_data.shape)
print(fe_data["gvkey"].nunique())

fe_gop = smf.ols(
    formula="""
    gop_w ~ 
        gop_L1
        + capx_at_L1
        + ppent_at_L1
        + emp_at_L1
        + xsga_at_L1
        + intan_at_L1
        + size_ln_at_L1
        + lev_w
        + liquidity_w
        + asset_turnover_w
        + firm_hhi_w
        + C(year)
        + C(gvkey)
    """,
    data=fe_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": fe_data["gvkey"]}
)

print(fe_gop.summary())

(1064, 14)
137
                            OLS Regression Results                            
Dep. Variable:                  gop_w   R-squared:                       0.937
Model:                            OLS   Adj. R-squared:                  0.926
Method:                 Least Squares   F-statistic:                     80.56
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           3.34e-69
Time:                        21:28:41   Log-Likelihood:                -7445.8
No. Observations:                1064   AIC:                         1.522e+04
Df Residuals:                     901   BIC:                         1.603e+04
Df Model:                         162                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 162, but rank is 25
  warnings.warn('covariance of constraints does not have full '


In [69]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = fe_gop.model.exog
vif = pd.DataFrame()
vif["variable"] = fe_gop.model.exog_names
vif["VIF"] = [variance_inflation_factor(X, i)
              for i in range(X.shape[1])]

vif.sort_values("VIF", ascending=False).head(20)

,variable,VIF
63,C(gvkey)[T.18213],2629.148760
156,xsga_at_L1,2038.540575
0,Intercept,1681.150928
153,capx_at_L1,720.705261
158,size_ln_at_L1,46.497255
6,C(year)[T.2016],15.973333
5,C(year)[T.2015],15.866684
7,C(year)[T.2017],15.845635
4,C(year)[T.2014],14.955972
3,C(year)[T.2013],14.843488


In [72]:
#로그 반영

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

In [73]:
# raw/log FE용 데이터셋: final_df에서 다시 시작
rawlog_df = final_df.copy()

print(rawlog_df.shape)
rawlog_df.head()

(1769, 107)


,gvkey,conm,tic,year,datadate,industry3,naics,sic,gop,ni_dv,...,ppent_at_w,emp_at_w,emp_sale_w,xsga_at_w,intan_at_w,lev_w,liquidity_w,asset_turnover_w,z_score_w,firm_hhi_w
0,102089,SODEXO S A,SDXAY,2010,2010-08-31,722,722310,5812,1267.859,519.594,...,0.045096,0.025345,0.019547,0.139788,0.438302,0.232951,0.954760,1.296645,2.101643,0.077362
1,102089,SODEXO S A,SDXAY,2011,2011-08-31,722,722310,5812,1568.813,649.711,...,0.044965,0.023799,0.016920,0.146463,0.418529,0.213603,0.940488,1.406521,2.290123,0.081876
2,102089,SODEXO S A,SDXAY,2012,2012-08-31,722,722310,5812,1612.501,660.345,...,0.044879,0.026194,0.018371,0.146052,0.437373,0.211181,0.936825,1.425801,2.405823,0.077811
3,102089,SODEXO S A,SDXAY,2013,2013-08-31,722,722310,5812,1576.922,579.304,...,0.042830,0.025720,0.017627,0.147763,0.422827,0.209946,0.869117,1.459153,2.425986,0.085320
4,102089,SODEXO S A,SDXAY,2014,2014-08-31,722,722310,5812,1526.715,644.350,...,0.038334,0.022008,0.017686,0.128056,0.379541,0.270272,1.026794,1.244371,2.255734,0.079843


In [74]:
def safe_log1p_nonneg(series):
    s = pd.to_numeric(series, errors="coerce")
    s = s.clip(lower=0)
    return np.log1p(s)

# raw explanatory variables
rawlog_df["ln_capx_raw"]  = safe_log1p_nonneg(rawlog_df["capx_raw"])
rawlog_df["ln_ppent_raw"] = safe_log1p_nonneg(rawlog_df["ppent_raw"])
rawlog_df["ln_emp_raw"]   = safe_log1p_nonneg(rawlog_df["emp_raw"])
rawlog_df["ln_xsga_raw"]  = safe_log1p_nonneg(rawlog_df["xsga_raw"])
rawlog_df["ln_intan_raw"] = safe_log1p_nonneg(rawlog_df["intan_raw"])

# firm size
# final_df에는 size_ln_at가 이미 있지만 raw/log 흐름을 명확히 하기 위해 다시 생성
rawlog_df["ln_at"] = safe_log1p_nonneg(rawlog_df["size_ln_at"].apply(np.exp) if False else rawlog_df["capx_raw"]*0 + np.nan)

In [75]:
raw_data = modeldata_gop[raw_log_vars].dropna().copy()

print("Rows after dropna:", raw_data.shape[0])
print("Unique gvkey:", raw_data["gvkey"].nunique())
print("Unique year:", raw_data["year"].nunique())

print("\nYear distribution:")
print(raw_data["year"].value_counts().sort_index())

Rows after dropna: 1055
Unique gvkey: 135
Unique year: 16

Year distribution:
year
2010     7
2011    71
2012    70
2013    77
2014    78
2015    83
2016    85
2017    81
2018    78
2019    73
2020    69
2021    68
2022    70
2023    73
2024    71
2025     1
Name: count, dtype: int64


In [76]:
# ln_at 대신 이미 만들어진 size_ln_at 사용
rawlog_df["ln_at"] = rawlog_df["size_ln_at"]

In [77]:
rawlog_df[[
    "ln_capx_raw", "ln_ppent_raw", "ln_emp_raw",
    "ln_xsga_raw", "ln_intan_raw", "ln_at"
]].describe().T

,count,mean,std,min,25%,50%,75%,max
ln_capx_raw,1711.0,3.330414,1.950021,0.000000,1.904638,3.478313,4.728183,8.022962
ln_ppent_raw,1715.0,5.296561,2.442563,0.000000,3.922537,5.525453,6.989794,10.565838
ln_emp_raw,1558.0,2.249379,1.556392,0.000000,1.098612,2.089513,3.198438,6.291569
ln_xsga_raw,1598.0,4.180898,1.895017,0.000000,2.825653,4.293318,5.513984,8.615046
ln_intan_raw,1674.0,3.960278,2.741811,0.000000,1.679597,4.221395,5.801179,9.810275
ln_at,1713.0,6.321491,2.355462,-6.214608,5.016670,6.496856,7.918284,10.935725


In [78]:
rawlog_df = rawlog_df.sort_values(["gvkey", "year"]).copy()

log_vars = [
    "ln_capx_raw",
    "ln_ppent_raw",
    "ln_emp_raw",
    "ln_xsga_raw",
    "ln_intan_raw",
    "ln_at"
]

for v in log_vars:
    rawlog_df[f"{v}_L1"] = rawlog_df.groupby("gvkey")[v].shift(1)

In [79]:
rawlog_df[[
    "gvkey", "year",
    "ln_capx_raw", "ln_capx_raw_L1",
    "ln_ppent_raw", "ln_ppent_raw_L1"
]].head(15)

,gvkey,year,ln_capx_raw,ln_capx_raw_L1,ln_ppent_raw,ln_ppent_raw_L1
0,102089,2010,5.706495,NaN,6.515575,NaN
1,102089,2011,5.856861,5.706495,6.606688,6.515575
2,102089,2012,5.962041,5.856861,6.583377,6.606688
3,102089,2013,5.765266,5.962041,6.570300,6.583377
4,102089,2014,5.778194,5.765266,6.594174,6.570300
5,102089,2015,5.826174,5.778194,6.501176,6.594174
6,102089,2016,5.954637,5.826174,6.513554,6.501176
7,102089,2017,5.909509,5.954637,6.554996,6.513554
8,102089,2018,5.947790,5.909509,6.578612,6.554996
9,102089,2019,6.088979,5.947790,6.624531,6.578612


In [80]:
rawlog_vars = [
    "gop_w", "gop_L1",
    "ln_capx_raw_L1", "ln_ppent_raw_L1", "ln_emp_raw_L1", "ln_xsga_raw_L1", "ln_intan_raw_L1",
    "ln_at_L1",
    "lev_w", "liquidity_w", "asset_turnover_w", "firm_hhi_w",
    "year", "gvkey"
]

print("Missing counts:")
print(rawlog_df[rawlog_vars].isna().sum().sort_values(ascending=False))

rawlog_data = rawlog_df[rawlog_vars].dropna().copy()

print("\nRows after dropna:", rawlog_data.shape[0])
print("Unique gvkey:", rawlog_data["gvkey"].nunique())
print("Unique year:", rawlog_data["year"].nunique())

Missing counts:
ln_emp_raw_L1       393
ln_xsga_raw_L1      359
ln_intan_raw_L1     289
gop_L1              257
ln_capx_raw_L1      254
ln_at_L1            253
ln_ppent_raw_L1     252
liquidity_w         167
gop_w                60
asset_turnover_w     59
lev_w                56
firm_hhi_w            0
year                  0
gvkey                 0
dtype: int64

Rows after dropna: 1171
Unique gvkey: 157
Unique year: 16


In [81]:
rawlog_fe_gop = smf.ols(
    formula="""
    gop_w ~
        gop_L1
        + ln_capx_raw_L1
        + ln_ppent_raw_L1
        + ln_emp_raw_L1
        + ln_xsga_raw_L1
        + ln_intan_raw_L1
        + ln_at_L1
        + lev_w
        + liquidity_w
        + asset_turnover_w
        + firm_hhi_w
        + C(year)
        + C(gvkey)
    """,
    data=rawlog_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": rawlog_data["gvkey"]}
)

In [82]:
print(rawlog_fe_gop.summary())

                            OLS Regression Results                            
Dep. Variable:                  gop_w   R-squared:                       0.937
Model:                            OLS   Adj. R-squared:                  0.926
Method:                 Least Squares   F-statistic:                     25.74
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           3.83e-43
Time:                        21:36:49   Log-Likelihood:                -8151.7
No. Observations:                1171   AIC:                         1.667e+04
Df Residuals:                     988   BIC:                         1.760e+04
Df Model:                         182                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            150.4236    302

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 182, but rank is 25
  warnings.warn('covariance of constraints does not have full '


In [83]:
rawlog_coef_table = pd.DataFrame({
    "coef": rawlog_fe_gop.params,
    "se": rawlog_fe_gop.bse,
    "pvalue": rawlog_fe_gop.pvalues
})

main_vars = [
    "gop_L1",
    "ln_capx_raw_L1",
    "ln_ppent_raw_L1",
    "ln_emp_raw_L1",
    "ln_xsga_raw_L1",
    "ln_intan_raw_L1",
    "ln_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w"
]

rawlog_coef_table.loc[main_vars]

,coef,se,pvalue
gop_L1,0.412450,0.092336,0.000008
ln_capx_raw_L1,-36.081291,21.297313,0.090233
ln_ppent_raw_L1,49.346921,39.446590,0.210942
ln_emp_raw_L1,-35.211983,66.116192,0.594326
ln_xsga_raw_L1,-12.406026,39.287128,0.752171
ln_intan_raw_L1,14.919432,27.398979,0.586080
ln_at_L1,73.347420,43.916086,0.094885
lev_w,-72.713714,53.374662,0.173095
liquidity_w,-6.203550,20.654451,0.763911
asset_turnover_w,28.226708,29.476926,0.338271


In [84]:
compare_table2 = pd.DataFrame({
    "scaled_coef": fe_gop.params.reindex([
        "gop_L1", "capx_at_L1", "ppent_at_L1", "emp_at_L1", "xsga_at_L1", "intan_at_L1",
        "lev_w", "liquidity_w", "asset_turnover_w", "firm_hhi_w"
    ]),
    "scaled_p": fe_gop.pvalues.reindex([
        "gop_L1", "capx_at_L1", "ppent_at_L1", "emp_at_L1", "xsga_at_L1", "intan_at_L1",
        "lev_w", "liquidity_w", "asset_turnover_w", "firm_hhi_w"
    ]),
    "rawlog_coef": rawlog_fe_gop.params.reindex([
        "gop_L1", "ln_capx_raw_L1", "ln_ppent_raw_L1", "ln_emp_raw_L1", "ln_xsga_raw_L1", "ln_intan_raw_L1",
        "lev_w", "liquidity_w", "asset_turnover_w", "firm_hhi_w"
    ]).values,
    "rawlog_p": rawlog_fe_gop.pvalues.reindex([
        "gop_L1", "ln_capx_raw_L1", "ln_ppent_raw_L1", "ln_emp_raw_L1", "ln_xsga_raw_L1", "ln_intan_raw_L1",
        "lev_w", "liquidity_w", "asset_turnover_w", "firm_hhi_w"
    ]).values
})

compare_table2

,scaled_coef,scaled_p,rawlog_coef,rawlog_p
gop_L1,0.404444,0.000013,0.412450,0.000008
capx_at_L1,-211.833852,0.117721,-36.081291,0.090233
ppent_at_L1,-27.120499,0.775990,49.346921,0.210942
emp_at_L1,-763.999539,0.545103,-35.211983,0.594326
xsga_at_L1,65.080458,0.367508,-12.406026,0.752171
intan_at_L1,-167.404905,0.077936,14.919432,0.586080
lev_w,-70.964843,0.134749,-72.713714,0.173095
liquidity_w,-22.332613,0.350203,-6.203550,0.763911
asset_turnover_w,30.220141,0.290346,28.226708,0.338271
firm_hhi_w,1408.922267,0.449163,1212.348270,0.468414


In [85]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif2 = rawlog_data[[
    "gop_L1",
    "ln_capx_raw_L1",
    "ln_ppent_raw_L1",
    "ln_emp_raw_L1",
    "ln_xsga_raw_L1",
    "ln_intan_raw_L1",
    "ln_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w"
]].dropna().copy()

vif_df2 = pd.DataFrame()
vif_df2["variable"] = X_vif2.columns
vif_df2["VIF"] = [variance_inflation_factor(X_vif2.values, i) for i in range(X_vif2.shape[1])]

vif_df2.sort_values("VIF", ascending=False)

,variable,VIF
6,ln_at_L1,85.915249
2,ln_ppent_raw_L1,82.521943
4,ln_xsga_raw_L1,56.056117
1,ln_capx_raw_L1,45.477047
10,firm_hhi_w,20.363650
3,ln_emp_raw_L1,16.470533
5,ln_intan_raw_L1,11.605456
9,asset_turnover_w,4.303046
8,liquidity_w,3.504635
7,lev_w,2.584798


In [86]:
raw_reduced_fe = smf.ols(
    formula="""
    gop_w ~

        gop_L1

        + ln_capx_raw_L1
        + ln_emp_raw_L1
        + ln_intan_raw_L1
        + ln_at_L1

        + lev_w
        + liquidity_w
        + asset_turnover_w
        + firm_hhi_w

        + C(year)
        + C(gvkey)
    """,
    data=rawlog_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": rawlog_data["gvkey"]}
)

In [87]:
raw_reduced_fe.summary()

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 180, but rank is 23
  warnings.warn('covariance of constraints does not have full '


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  gop_w   R-squared:                       0.937
Model:                            OLS   Adj. R-squared:                  0.926
Method:                 Least Squares   F-statistic:                     56.91
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           5.45e-64
Time:                        21:45:01   Log-Likelihood:                -8153.3
No. Observations:                1171   AIC:                         1.667e+04
Df Residuals:                     990   BIC:                         1.759e+04
Df Model:                         180                                         
Covariance Type:              cluster                                         
======================================================================================
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             15.7768    315.297      0.050      0.960    -602.194     633.748
C(year)[T.2011]      -23.2484     26.260     -0.885      0.376     -74.717      28.220
C(year)[T.2012]      -23.5828     31.465     -0.750      0.454     -85.252      38.087
C(year)[T.2013]       -2.6131     25.349     -0.103      0.918     -52.297      47.071
C(year)[T.2014]       10.1335     26.552      0.382      0.703     -41.907      62.174
C(year)[T.2015]       -3.2479     35.962     -0.090      0.928     -73.731      67.235
C(year)[T.2016]       -5.6912     36.093     -0.158      0.875     -76.432      65.049
C(year)[T.2017]       29.1337     37.686      0.773      0.439     -44.729     102.997
C(year)[T.2018]       26.5569     32.855      0.808      0.419     -37.838      90.952
C(year)[T.2019]       38.7379     33.360      1.161      0.246     -26.647     104.123
C(year)[T.2020]     -300.4261    120.270     -2.498      0.012    -536.151     -64.701
C(year)[T.2021]       22.1013     84.890      0.260      0.795    -144.280     188.483
C(year)[T.2022]      -35.6984     80.901     -0.441      0.659    -194.262     122.865
C(year)[T.2023]       59.6279     47.667      1.251      0.211     -33.798     153.054
C(year)[T.2024]       43.8370     60.059      0.730      0.465     -73.877     161.551
C(year)[T.2025]      261.9213     77.529      3.378      0.001     109.967     413.875
C(gvkey)[T.106660]  -552.2355    270.347     -2.043      0.041   -1082.106     -22.365
C(gvkey)[T.107148]  -360.6770    283.877     -1.271      0.204    -917.065     195.711
C(gvkey)[T.108331]  -449.8844    244.603     -1.839      0.066    -929.298      29.530
C(gvkey)[T.109079]  -638.6414    248.259     -2.572      0.010   -1125.221    -152.062
C(gvkey)[T.11538]   -531.0669    230.258     -2.306      0.021    -982.365     -79.769
C(gvkey)[T.116503]  -589.8869    189.126     -3.119      0.002    -960.567    -219.207
C(gvkey)[T.117036]  -594.0527    210.654     -2.820      0.005   -1006.927    -181.178
C(gvkey)[T.11872]   -525.0648    240.389     -2.184      0.029    -996.219     -53.911
C(gvkey)[T.130307]  -405.9939    275.139     -1.476      0.140    -945.256     133.268
C(gvkey)[T.13092]   -530.9403    159.676     -3.325      0.001    -843.899    -217.982
C(gvkey)[T.133767]  -643.0560    189.602     -3.392      0.001   -1014.668    -271.444
C(gvkey)[T.134504]  -304.9305    272.019     -1.121      0.262    -838.078     228.217
C(gvkey)[T.13759]   -538.3244    207.032     -2.600      0.009    -944.101    -132.548
C(gvkey)[T.138861]  -700.2394    198.065     -3.535      0.000   -1088.440    -312.039
C(gvkey)[T.14418]    231.9567    107.532      2.157      0.031      21.198     442.716
C(gvkey)[T.144519]  -251.5597     65.485     -3.841      0.000    -379.908    -123.211
C(gvkey)[T.148249]  -446.2850    265.3

In [90]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif2 = rawlog_df[[
    "gop_L1",
    "ln_capx_raw_L1",
    "ln_emp_raw_L1",
    "ln_xsga_raw_L1",
    "ln_intan_raw_L1",
    "ln_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w"
]].dropna().copy()

vif_df2 = pd.DataFrame()
vif_df2["variable"] = X_vif2.columns
vif_df2["VIF"] = [variance_inflation_factor(X_vif2.values, i) for i in range(X_vif2.shape[1])]

vif_df2.sort_values("VIF", ascending=False)

,variable,VIF
5,ln_at_L1,71.433861
3,ln_xsga_raw_L1,52.109297
1,ln_capx_raw_L1,30.003917
9,firm_hhi_w,18.912881
2,ln_emp_raw_L1,16.420974
4,ln_intan_raw_L1,10.815239
8,asset_turnover_w,4.119972
7,liquidity_w,3.211126
6,lev_w,2.580346
0,gop_L1,1.711565


In [91]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_scaled = modeldata_gop[[
    "gop_L1",
    "capx_at_L1",
    "ppent_at_L1",
    "emp_at_L1",
    "xsga_at_L1",
    "intan_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w"
]].dropna()

vif_scaled = pd.DataFrame()
vif_scaled["variable"] = X_scaled.columns
vif_scaled["VIF"] = [
    variance_inflation_factor(X_scaled.values, i)
    for i in range(X_scaled.shape[1])
]

vif_scaled.sort_values("VIF", ascending=False)

,variable,VIF
4,xsga_at_L1,184.227285
1,capx_at_L1,182.593785
9,firm_hhi_w,21.905438
2,ppent_at_L1,9.477562
8,asset_turnover_w,4.516727
7,liquidity_w,3.565059
5,intan_at_L1,2.788040
3,emp_at_L1,2.655284
6,lev_w,2.587229
0,gop_L1,1.221175


In [92]:
import numpy as np
import pandas as pd

# 정렬 (필수)
df = df.sort_values(["gvkey", "fyear"])

# ----------------------------
# 1️⃣ Lag 생성
# ----------------------------

lag_vars = [
    "gop",
    "capx_raw",
    "emp_raw",
    "xsga_raw",
    "intan_raw",
    "ppent_raw",
    "at"
]

for v in lag_vars:
    df[f"{v}_L1"] = df.groupby("gvkey")[v].shift(1)

# ----------------------------
# 2️⃣ Ratio 생성 (size 제거)
# ----------------------------

df["capx_at_L1"]   = df["capx_raw_L1"]   / df["at_L1"]
df["emp_at_L1"]    = df["emp_raw_L1"]    / df["at_L1"]
df["xsga_at_L1"]   = df["xsga_raw_L1"]   / df["at_L1"]
df["intan_at_L1"]  = df["intan_raw_L1"]  / df["at_L1"]

# robustness용
df["ppent_at_L1"]  = df["ppent_raw_L1"]  / df["at_L1"]

# GOP lag
df["gop_L1"] = df["gop_L1"]

In [96]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

work_df = final_df.copy()
work_df = work_df.sort_values(["gvkey", "year"]).copy()

In [97]:
def winsorize_series(s, lower=0.01, upper=0.99):
    s = pd.to_numeric(s, errors="coerce")
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lower=lo, upper=hi)

scaled_vars = [
    "capx_at_L1",
    "ppent_at_L1",
    "emp_at_L1",
    "xsga_at_L1",
    "intan_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w"
]

for v in scaled_vars:
    work_df[v] = pd.to_numeric(work_df[v], errors="coerce")
    work_df[v] = work_df[v].replace([np.inf, -np.inf], np.nan)
    work_df[v] = winsorize_series(work_df[v], 0.01, 0.99)

In [98]:
op_vars = ["emp_at_L1", "xsga_at_L1"]
op_df = work_df[op_vars].dropna().copy()

scaler_op = StandardScaler()
op_scaled = scaler_op.fit_transform(op_df)

pca_op = PCA(n_components=1)
op_pc1 = pca_op.fit_transform(op_scaled)

work_df.loc[op_df.index, "operational_pc1"] = op_pc1[:, 0]

print("Operational PCA explained variance ratio:")
print(pca_op.explained_variance_ratio_)

Operational PCA explained variance ratio:
[0.65390639]


In [99]:
phys_vars = ["capx_at_L1", "ppent_at_L1"]
phys_df = work_df[phys_vars].dropna().copy()

scaler_phys = StandardScaler()
phys_scaled = scaler_phys.fit_transform(phys_df)

pca_phys = PCA(n_components=1)
phys_pc1 = pca_phys.fit_transform(phys_scaled)

work_df.loc[phys_df.index, "physical_pc1"] = phys_pc1[:, 0]

print("Physical PCA explained variance ratio:")
print(pca_phys.explained_variance_ratio_)

Physical PCA explained variance ratio:
[0.69496102]


In [100]:
main_vars = [
    "gop_w", "gop_L1",
    "capx_at_L1",
    "operational_pc1",
    "intan_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w",
    "year",
    "gvkey"
]

main_data = work_df[main_vars].dropna().copy()

print(main_data.shape)
print(main_data["gvkey"].nunique())
print(main_data["year"].nunique())

(1171, 11)
157
16


In [101]:
main_fe_gop = smf.ols(
    formula="""
    gop_w ~
        gop_L1
        + capx_at_L1
        + operational_pc1
        + intan_at_L1
        + lev_w
        + liquidity_w
        + asset_turnover_w
        + firm_hhi_w
        + C(year)
        + C(gvkey)
    """,
    data=main_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": main_data["gvkey"]}
)

print(main_fe_gop.summary())

                            OLS Regression Results                            
Dep. Variable:                  gop_w   R-squared:                       0.936
Model:                            OLS   Adj. R-squared:                  0.925
Method:                 Least Squares   F-statistic:                     113.1
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           3.31e-84
Time:                        22:12:21   Log-Likelihood:                -8162.9
No. Observations:                1171   AIC:                         1.669e+04
Df Residuals:                     991   BIC:                         1.760e+04
Df Model:                         179                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            842.2901    140

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 179, but rank is 22
  warnings.warn('covariance of constraints does not have full '


In [102]:
main_coef_table = pd.DataFrame({
    "coef": main_fe_gop.params,
    "se": main_fe_gop.bse,
    "pvalue": main_fe_gop.pvalues
})

print(main_coef_table.loc[
    ["gop_L1", "capx_at_L1", "operational_pc1", "intan_at_L1",
     "lev_w", "liquidity_w", "asset_turnover_w", "firm_hhi_w"]
])

                        coef           se    pvalue
gop_L1              0.423953     0.093625  0.000006
capx_at_L1       -236.044499   154.605979  0.126823
operational_pc1   -21.913748    20.820473  0.292566
intan_at_L1         1.847158    84.369069  0.982533
lev_w             -57.955592    52.030946  0.265336
liquidity_w       -18.198289    17.734485  0.304820
asset_turnover_w   16.012577    26.620280  0.547495
firm_hhi_w        651.351879  1746.123217  0.709128


In [103]:
X_main_vif = main_data[[
    "gop_L1",
    "capx_at_L1",
    "operational_pc1",
    "intan_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w"
]].copy()

vif_main = pd.DataFrame()
vif_main["variable"] = X_main_vif.columns
vif_main["VIF"] = [
    variance_inflation_factor(X_main_vif.values, i)
    for i in range(X_main_vif.shape[1])
]

print(vif_main.sort_values("VIF", ascending=False))

           variable       VIF
7        firm_hhi_w  9.815924
6  asset_turnover_w  4.740510
5       liquidity_w  3.086942
1        capx_at_L1  2.851082
4             lev_w  2.332588
3       intan_at_L1  1.885325
2   operational_pc1  1.695417
0            gop_L1  1.220643


In [104]:
robust_vars = [
    "gop_w", "gop_L1",
    "physical_pc1",
    "operational_pc1",
    "intan_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w",
    "year",
    "gvkey"
]

robust_data = work_df[robust_vars].dropna().copy()

robust_fe_gop = smf.ols(
    formula="""
    gop_w ~
        gop_L1
        + physical_pc1
        + operational_pc1
        + intan_at_L1
        + lev_w
        + liquidity_w
        + asset_turnover_w
        + firm_hhi_w
        + C(year)
        + C(gvkey)
    """,
    data=robust_data
).fit(
    cov_type="cluster",
    cov_kwds={"groups": robust_data["gvkey"]}
)

print(robust_fe_gop.summary())

                            OLS Regression Results                            
Dep. Variable:                  gop_w   R-squared:                       0.936
Model:                            OLS   Adj. R-squared:                  0.925
Method:                 Least Squares   F-statistic:                     81.51
Date:                Wed, 01 Apr 2026   Prob (F-statistic):           5.65e-74
Time:                        22:13:16   Log-Likelihood:                -8163.4
No. Observations:                1171   AIC:                         1.669e+04
Df Residuals:                     991   BIC:                         1.760e+04
Df Model:                         179                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            816.1528    144

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 179, but rank is 22
  warnings.warn('covariance of constraints does not have full '


In [105]:
import os
import pandas as pd

# 저장 경로
OUTPUT_DIR = "/Volumes/ORICO/Dissertation/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 저장용 데이터프레임: work_df 기준
export_df = work_df.copy()

# =========================================================
# 저장할 주요 변수 목록
# =========================================================
base_cols = [
    # ID
    "gvkey", "conm", "tic", "year", "datadate", "industry3", "naics", "sic",

    # Dependent variables
    "gop", "gop_w", "gop_L1",
    "ni_dv", "ni_dv_w", "ni_dv_L1",
    "tobinq", "tobinq_w", "tobinq_L1",
    "roa", "roa_w", "roa_L1",
    "ros", "ros_w", "ros_L1",
    "operating_margin", "operating_margin_w", "operating_margin_L1",
    "sales_growth", "sales_growth_w", "sales_growth_L1",

    # Raw resource variables
    "capx_raw", "capx_raw_L1",
    "ppent_raw", "ppent_raw_L1",
    "emp_raw", "emp_raw_L1",
    "xsga_raw", "xsga_raw_L1",
    "intan_raw", "intan_raw_L1",

    # Ratio-scaled lagged variables
    "capx_at_L1",
    "ppent_at_L1",
    "emp_at_L1",
    "xsga_at_L1",
    "intan_at_L1",

    # PCA bundle variables
    "operational_pc1",
    "physical_pc1",

    # Controls
    "size_ln_at", "size_ln_at_L1",
    "lev", "lev_w", "lev_L1",
    "liquidity", "liquidity_w", "liquidity_L1",
    "asset_turnover", "asset_turnover_w", "asset_turnover_L1",
    "z_score", "z_score_w", "z_score_L1",
    "distress_dummy", "distress_dummy_L1",
    "firm_hhi", "firm_hhi_w", "firm_hhi_L1",
    "market_share"
]

# 실제 존재하는 컬럼만 선택
export_cols = [c for c in base_cols if c in export_df.columns]
export_df = export_df[export_cols].copy()

print("Export shape:", export_df.shape)
print("Number of columns:", len(export_df.columns))
print(export_df.columns.tolist())

Export shape: (1769, 66)
Number of columns: 66
['gvkey', 'conm', 'tic', 'year', 'datadate', 'industry3', 'naics', 'sic', 'gop', 'gop_w', 'gop_L1', 'ni_dv', 'ni_dv_w', 'ni_dv_L1', 'tobinq', 'tobinq_w', 'tobinq_L1', 'roa', 'roa_w', 'roa_L1', 'ros', 'ros_w', 'ros_L1', 'operating_margin', 'operating_margin_w', 'operating_margin_L1', 'sales_growth', 'sales_growth_w', 'sales_growth_L1', 'capx_raw', 'capx_raw_L1', 'ppent_raw', 'ppent_raw_L1', 'emp_raw', 'emp_raw_L1', 'xsga_raw', 'xsga_raw_L1', 'intan_raw', 'intan_raw_L1', 'capx_at_L1', 'ppent_at_L1', 'emp_at_L1', 'xsga_at_L1', 'intan_at_L1', 'operational_pc1', 'physical_pc1', 'size_ln_at', 'size_ln_at_L1', 'lev', 'lev_w', 'lev_L1', 'liquidity', 'liquidity_w', 'liquidity_L1', 'asset_turnover', 'asset_turnover_w', 'asset_turnover_L1', 'z_score', 'z_score_w', 'z_score_L1', 'distress_dummy', 'distress_dummy_L1', 'firm_hhi', 'firm_hhi_w', 'firm_hhi_L1', 'market_share']


In [106]:
csv_path = f"{OUTPUT_DIR}/dissertation_cleaned_with_pca_bundles.csv"
xlsx_path = f"{OUTPUT_DIR}/dissertation_cleaned_with_pca_bundles.xlsx"

export_df.to_csv(csv_path, index=False)
export_df.to_excel(xlsx_path, index=False)

print("Saved CSV:", csv_path)
print("Saved Excel:", xlsx_path)

Saved CSV: /Volumes/ORICO/Dissertation/output/dissertation_cleaned_with_pca_bundles.csv
Saved Excel: /Volumes/ORICO/Dissertation/output/dissertation_cleaned_with_pca_bundles.xlsx


In [107]:
print(export_df[[
    c for c in [
        "gvkey", "year", "gop_w", "gop_L1",
        "capx_at_L1", "emp_at_L1", "xsga_at_L1", "intan_at_L1",
        "operational_pc1", "physical_pc1",
        "lev_w", "liquidity_w", "asset_turnover_w", "firm_hhi_w"
    ] if c in export_df.columns
]].head(10))

    gvkey  year     gop_w    gop_L1  capx_at_L1  emp_at_L1  xsga_at_L1  \
0  102089  2010  1267.859       NaN         NaN        NaN         NaN   
1  102089  2011  1568.813  1267.859    0.020043   0.025345    0.139788   
2  102089  2012  1612.501  1568.813    0.021211   0.023799    0.146463   
3  102089  2013  1576.922  1612.501    0.024081   0.026194    0.146052   
4  102089  2014  1526.715  1576.922    0.019115   0.025720    0.147763   
5  102089  2015  1550.368  1526.715    0.016922   0.022008    0.128056   
6  102089  2016  1525.888  1550.368    0.020861   0.026093    0.141120   
7  102089  2017  1694.895  1525.888    0.024368   0.026970    0.144653   
8  102089  2018  1574.058  1694.895    0.020775   0.024152    0.137488   
9  102089  2019  1593.795  1574.058    0.021531   0.025972    0.132788   

   intan_at_L1  operational_pc1  physical_pc1     lev_w  liquidity_w  \
0          NaN              NaN           NaN  0.232951     0.954760   
1     0.438302         0.194299     -1.61

In [108]:
main_export_vars = [
    "gvkey", "year", "gop_w", "gop_L1",
    "capx_at_L1",
    "operational_pc1",
    "intan_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w"
]

main_export_vars = [c for c in main_export_vars if c in work_df.columns]
main_export_df = work_df[main_export_vars].dropna().copy()

main_csv = f"{OUTPUT_DIR}/modeldata_main_pca_bundle.csv"
main_export_df.to_csv(main_csv, index=False)

print("Saved main model file:", main_csv)
print("Main model shape:", main_export_df.shape)

Saved main model file: /Volumes/ORICO/Dissertation/output/modeldata_main_pca_bundle.csv
Main model shape: (1171, 11)


In [109]:
robust_export_vars = [
    "gvkey", "year", "gop_w", "gop_L1",
    "physical_pc1",
    "operational_pc1",
    "intan_at_L1",
    "lev_w",
    "liquidity_w",
    "asset_turnover_w",
    "firm_hhi_w"
]

robust_export_vars = [c for c in robust_export_vars if c in work_df.columns]
robust_export_df = work_df[robust_export_vars].dropna().copy()

robust_csv = f"{OUTPUT_DIR}/modeldata_robustness_pca_bundle.csv"
robust_export_df.to_csv(robust_csv, index=False)

print("Saved robustness model file:", robust_csv)
print("Robustness model shape:", robust_export_df.shape)

Saved robustness model file: /Volumes/ORICO/Dissertation/output/modeldata_robustness_pca_bundle.csv
Robustness model shape: (1171, 11)
